# 03. Eficiência Agrícola - Potencial de Melhoria

**Objetivo:** Identificar municípios com potencial de melhoria de eficiência agrícola para políticas públicas de desenvolvimento sustentável.

**Impacto no Negócio:** Orientar políticas públicas e programas de capacitação para municípios com potencial de otimização econômica sustentável.

**Dados de Entrada:** `data/04_modelagem/dataset_preditivo_com_precos.parquet`

**Dados de Saída:** `data/03_gold/potencial_melhoria_eficiencia_2023.parquet`

In [ ]:
# ============================================================================
# MONTAR GOOGLE DRIVE (APENAS COLAB)
# ============================================================================

def montar_google_drive():
    """Monta o Google Drive no Colab."""
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        print("✓ Google Drive montado em /content/drive")
        return True
    except Exception as e:
        print(f"⚠️  Erro ao montar Google Drive: {e}")
        return False

# Detecta se está no Colab e tenta montar o Drive
try:
    import google.colab
    print("📤 Ambiente Google Colab detectado")
    print("Montando Google Drive...")
    montar_google_drive()
except ImportError:
    print("✓ Ambiente local detectado - não é necessário montar Drive")

In [ ]:
# ============================================================================
# CONFIGURAÇÃO DE AMBIENTE
# ============================================================================

import sys
import os
from pathlib import Path

# Detectar ambiente e configurar caminho corretamente
try:
    import google.colab
    print("📤 Ambiente Google Colab detectado")
    # No Colab, usar o diretório do drive
    if os.path.exists('/content/drive/MyDrive/dados_analise'):
        os.chdir('/content/drive/MyDrive/dados_analise')
        print("✓ Diretório alterado para: /content/drive/MyDrive/dados_analise")
    else:
        print("⚠️  Diretório dados_analise não encontrado no Drive")
except ImportError:
    print("✓ Ambiente local detectado")
    # Local, usar diretório atual
    current_dir = Path.cwd()
    # Se estiver em notebooks_analise_preditiva, voltar para o root
    if 'notebooks_analise_preditiva' in str(current_dir):
        os.chdir(current_dir.parent)
        print(f"✓ Diretório alterado para: {current_dir.parent}")

print(f"✓ Diretório de trabalho atual: {os.getcwd()}")

# ============================================================================
# CONFIGURAÇÃO DE CAMINHOS
# ============================================================================

# Caminho direto para os dados (já está no drive)
CAMINHO_DADOS = 'data/04_modelagem/dataset_preditivo_com_precos.parquet'
CAMINHO_SAIDA = 'data/03_gold/potencial_melhoria_eficiencia_2023.parquet'

print(f"\nCaminho dos dados: {CAMINHO_DADOS}")
print(f"Caminho de saída: {CAMINHO_SAIDA}")

# Verificar se o arquivo existe
if os.path.exists(CAMINHO_DADOS):
    print(f"✓ Arquivo de dados encontrado")
else:
    print(f"⚠️  Arquivo de dados não encontrado: {CAMINHO_DADOS}")

In [ ]:
## 1. Configuração e Importações
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURAÇÃO
# ============================================================================

# Configurações específicas para este notebook
ANO_ATUAL = 2023
ANO_PROJECAO = 2024

# Thresholds para classificação de tendências (ha/ano)
THRESHOLD_AUMENTO_FORTE = 10
THRESHOLD_AUMENTO_LEVE = 0
THRESHOLD_REDUCAO_LEVE = -10
THRESHOLD_REDUCAO_FORTE = -50

CATEGORIAS_TENDENCIA = ['Redução Forte', 'Redução Leve', 'Estável', 'Aumento Leve', 'Aumento Forte']

# UFs da Amazônia Legal
UFS_AMAZONIA_LEGAL = ['AC', 'AM', 'AP', 'MA', 'MT', 'PA', 'RO', 'RR', 'TO']

In [ ]:
# ============================================================================
# PASSO 1: CARREGAMENTO E VALIDAÇÃO DOS DADOS
# ============================================================================

print("PASSO 1: Carregando e validando dados...")

# Carrega o dataset principal
df_completo = carregar_dados(CAMINHO_DADOS)

print(f"✓ Dataset carregado: {df_completo.shape[0]:,} observações × {df_completo.shape[1]} colunas")
print(f"✓ Período: {df_completo['ano'].min()} - {df_completo['ano'].max()}")
print(f"✓ Municípios únicos: {df_completo['cod_ibge'].nunique():,}")

In [ ]:
# ============================================================================
# PASSO 2: FILTRAGEM DA AMAZÔNIA LEGAL
# ============================================================================

print("\nPASSO 2: Filtrando dados para Amazônia Legal...")

# Filtra apenas municípios da Amazônia Legal
df_amazonia = filtrar_amazonia_legal(df_completo, UFS_AMAZONIA_LEGAL)

print(f"✓ Amazônia Legal: {df_amazonia.shape[0]:,} observações")
print(f"✓ Municípios na Amazônia Legal: {df_amazonia['cod_ibge'].nunique():,}")

In [ ]:
# ============================================================================
# PASSO 3: CÁLCULO DE EFICIÊNCIA AGRÍCOLA
# ============================================================================

print("\nPASSO 3: Calculando indicador de eficiência agrícola...")

# Calcula eficiência agrícola (VAB por produção de soja)
df_amazonia = calcular_eficiencia_agricola(df_amazonia)

# Cria target binário para alta eficiência
df_amazonia = criar_target_alta_eficiencia(df_amazonia, PERCENTIL_ALTA_EFICIENCIA)

# Recupera threshold para referência futura
threshold_eficiencia = df_amazonia['eficiencia_agricola'].quantile(PERCENTIL_ALTA_EFICIENCIA)

print(f"✓ Threshold de alta eficiência: {threshold_eficiencia:.2f}")
print(f"✓ Municípios com alta eficiência: {df_amazonia['alta_eficiencia'].sum():,} ({df_amazonia['alta_eficiencia'].mean()*100:.1f}%)")

In [ ]:
# ============================================================================
# PASSO 4: PREPARAÇÃO DAS FEATURES
# ============================================================================

print("\nPASSO 4: Preparando features para o modelo...")

# Prepara features e target
X, y, threshold_eficiencia = preparar_features_modelo(
    df_amazonia, 
    FEATURES_MODELO_EFICIENCIA, 
    TARGET_EFICIENCIA
)

print(f"✓ Dataset preparado: {X.shape[0]:,} observações × {X.shape[1]} features")
print(f"✓ Target: {TARGET_EFICIENCIA}")
print(f"✓ Threshold eficiência: {threshold_eficiencia:.2f}")
print(f"✓ Distribuição target: {y.mean()*100:.2f}% alta eficiência")

In [ ]:
# ============================================================================
# PASSO 5: DIVISÃO TEMPORAL DOS DADOS
# ============================================================================

print("\nPASSO 5: Dividindo dados temporalmente (treino/teste)...")

# Divide os dados em treino e teste baseado em critério temporal
X_train, X_test, y_train, y_test = dividir_dados_temporalmente(
    X, y, ANO_LIMITE_TREINO, ANO_TESTE
)

print(f"✓ Treino: {X_train.shape[0]:,} observações ({X_train['ano'].min()}-{X_train['ano'].max()})")
print(f"✓ Teste: {X_test.shape[0]:,} observações ({X_test['ano'].min()}-{X_test['ano'].max()})")
print(f"✓ Distribuição treino: {y_train.mean()*100:.2f}% alta eficiência")
print(f"✓ Distribuição teste: {y_test.mean()*100:.2f}% alta eficiência")

In [ ]:
# ============================================================================
# PASSO 6: TREINAMENTO DO MODELO
# ============================================================================

print("\nPASSO 6: Treinando modelo Random Forest...")

# Treina o modelo com os parâmetros otimizados
modelo_eficiencia = treinar_modelo_random_forest(X_train, y_train)

print("✓ Modelo treinado com sucesso!")
print(f"✓ Tipo: RandomForestClassifier")
print(f"✓ Estimators: 200")
print(f"✓ Max Depth: 6 (para evitar overfitting)")

In [ ]:
# ============================================================================
# PASSO 7: AVALIAÇÃO DO MODELO
# ============================================================================

print("\nPASSO 7: Avaliando desempenho do modelo...")

# Gera previsões probabilísticas
y_pred_proba = modelo_eficiencia.predict_proba(X_test)[:, 1]

# Encontra threshold ótimo para maximizar F1-score
threshold_otimo, precision, recall, f1_scores = otimizar_threshold(y_test, y_pred_proba)

# Aplica threshold otimizado
y_pred_otimizado = (y_pred_proba >= threshold_otimo).astype(int)

# Calcula métricas de avaliação
roc_auc = roc_auc_score(y_test, y_pred_proba)
pr_auc = auc(recall, precision)

print("✓ MÉTRICAS DE AVALIAÇÃO:")
print(f"  ROC-AUC: {roc_auc:.4f}")
print(f"  Precision-Recall AUC: {pr_auc:.4f}")
print(f"  Threshold ótimo: {threshold_otimo:.4f}")

print("\n✓ RELATÓRIO DE CLASSIFICAÇÃO:")
print(classification_report(y_test, y_pred_otimizado, 
                          target_names=['Baixa Eficiência', 'Alta Eficiência']))

In [ ]:
# ============================================================================
# PASSO 8: ANÁLISE DE IMPORTÂNCIA DAS FEATURES
# ============================================================================

print("\nPASSO 8: Analisando importância das features...")

# Cria DataFrame com importância das features
df_importancia = criar_feature_importance(modelo_eficiencia, FEATURES_MODELO_EFICIENCIA)

print("✓ TOP 10 FEATURES MAIS IMPORTANTES:")
print(df_importancia.head(10).to_string(index=False))

In [ ]:
# ============================================================================
# PASSO 9: IDENTIFICAÇÃO DE POTENCIAL DE MELHORIA
# ============================================================================

print("\nPASSO 9: Identificando municípios com potencial de melhoria...")

# Identifica municípios com baixa eficiência atual mas alto potencial
df_2022 = df_amazonia[df_amazonia['ano'] == ANO_TESTE].copy()

# Identifica potencial de melhoria
probabilidade_minima_potencial = 0.5
potencial_melhoria = identificar_potencial_melhoria(
    df_2022,
    modelo_eficiencia,
    FEATURES_MODELO_EFICIENCIA,
    threshold_eficiencia,
    probabilidade_minima_potencial
)

# Seleciona colunas relevantes para análise
colunas_analise = ['cod_ibge', 'municipio', 'uf', 'eficiencia_agricola',
                  'probabilidade_alta_eficiencia', 'vab_agro_mil_reais', 
                  'producao_soja_mil_ton']
potencial_melhoria = potencial_melhoria[colunas_analise]

print(f"✓ Municípios com potencial de melhoria: {len(potencial_melhoria)}")
print(f"\n✓ TOP 20 MUNICÍPIOS COM MAIOR POTENCIAL DE MELHORIA:")
print(potencial_melhoria.head(20).to_string(index=False))

In [ ]:
# ============================================================================
# PASSO 10: CÁLCULO DE ESTATÍSTICAS DE IMPACTO
# ============================================================================

print("\nPASSO 10: Calculando estatísticas de impacto...")

# Calcula estatísticas de impacto
estatisticas_impacto = calcular_estatisticas_impacto(potencial_melhoria)

print("✓ ESTATÍSTICAS DE IMPACTO DO POTENCIAL DE MELHORIA:")
print(f"  Quantidade de municípios: {estatisticas_impacto['quantidade_municipios']}")
print(f"  Probabilidade média de alta eficiência: {estatisticas_impacto['probabilidade_media']*100:.1f}%")
print(f"  VAB agropecuário total: R$ {estatisticas_impacto['vab_total']*1000:,.0f}")
print(f"  Produção de soja total: {estatisticas_impacto['producao_soja_total']:,.0f} mil toneladas")

In [ ]:
# ============================================================================
# PASSO 11: SALVAMENTO DOS RESULTADOS
# ============================================================================

print("\nPASSO 11: Salvando resultados...")

# Salva o potencial de melhoria
salvar_potencial_melhoria(potencial_melhoria, CAMINHO_SAIDA)

print(f"✓ Potencial de melhoria salvo em: {CAMINHO_SAIDA}")
print(f"✓ Total de municípios com potencial: {len(potencial_melhoria)}")

print("\n" + "="*70)
print("✅ ANÁLISE DE EFICIÊNCIA AGRÍCOLA CONCLUÍDA COM SUCESSO")
print("="*70)

# Célula removida - código antigo não usado

In [ ]:
# Célula removida - código antigo não usado

# Célula removida - código antigo não usado

In [ ]:
# Célula removida - código antigo não usado

# Célula removida - código antigo não usado

In [ ]:
# Célula removida - código antigo não usado

## 11. Salvamento dos Resultados

In [ ]:
# Salvar potencial de melhoria
potencial_melhoria.to_parquet(CAMINHO_SAIDA, index=False)
print(f'\nPotencial de melhoria salvo em {CAMINHO_SAIDA}')
print(f'Total de municípios com potencial: {len(potencial_melhoria)}')

## 12. Conclusão

**Resumo da Análise:**
- Modelo de Random Forest treinado com ROC-AUC de {roc_auc_score(y_test_efic, y_pred_proba_efic):.4f}
- {len(potencial_melhoria)} municípios identificados com potencial de melhoria de eficiência
- Probabilidade média de alta eficiência: {potencial_melhoria["probabilidade_alta_eficiencia"].mean()*100:.1f}%

**Impacto no Negócio:**
- Orientação para políticas públicas de desenvolvimento sustentável
- Identificação de municípios para programas de capacitação
- Potencial econômico de {potencial_melhoria["vab_agro_mil_reais"].sum()*1000:,.0f}

**Próximos Passos:**
- Desenvolver programas de capacitação técnica
- Implementar incentivos fiscais para práticas eficientes
- Criar parcerias com cooperativas agrícolas